# EDA 07: chọn ngưỡng lọc chuỗi outlier

**Mục đích:** chọn cách lọc chuỗi điểm GPS lỗi, theo hai cách và so sánh:
- **Cách N:** xoá chuỗi có số điểm `k ≤ N`.
- **Cách T:** xoá chuỗi có thời gian từ A đến B `dur_s ≤ T` (giây).

Notebook này không tự chọn ngưỡng. Nó tính các số liệu để bạn so sánh, và ghi tiêu chí chốt ở cuối.

**Chuỗi là gì.** Một đoạn gồm các điểm X lệch khỏi đường đi thật, nằm giữa hai điểm A và B đáng tin:

```
A ──► X1 ... Xk ──► B
```

Một chuỗi được ghi nhận khi cả bước vào (A→X1) và bước ra (Xk→B) đều nhanh hơn 300 km/h (mốc thử, xét ở notebook 06), khoảng cách thời gian mỗi bước không quá 30 phút (mốc thử), và cùng một file. Notebook đếm rộng (tối đa 50 điểm, A→B tối đa 2 giờ), rồi áp dụng N hoặc T lên tập này.

**Hai đại lượng:**
- `k`: số điểm X trong chuỗi.
- `dur_s`: thời gian từ A đến B, tính bằng giây.

**Điều kiện `back_ratio` đã bỏ khỏi phần quyết định.** Mọi bảng trong notebook này đếm trên toàn bộ chuỗi, để hai cách được so sánh trên cùng một tập.

**Cách chạy.** Chạy `python src/preprocess/convert_raw.py` một lần trước để tạo `data/interim`. Sau đó Run All, mất khoảng 1–2 phút. Muốn chạy thử thì đặt `USERS_LIMIT` ở cell đầu.

**Các bước:**
1. Tập chuỗi và phân bố cơ bản.
2. Lưới ứng viên N và T với các chỉ số cho từng ứng viên.
3. Tìm điểm gập: nơi số điểm bị xoá tăng đột ngột khi nới thêm một bậc.
4. So hai cách ở cùng số điểm bị xoá.
5. Bảng 2×2 cho cặp ngưỡng đã chọn.
6. Kiểm tra theo nhóm ghi dày và thưa.
7. Tiêu chí chốt (cuối notebook).


> **Giai đoạn: TUNING (sau khi chia validation / test).** Chỉ đọc **nhóm validation** (`split.val_users()`). Không dùng nhóm test (D17, D19).

In [9]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(ROOT / "src"))
from eda.config import *          # dữ kiện từ tài liệu, mốc mô tả, bins (xem src/eda/config.py)
from eda.utils import fmt_s
from eda.scan import run
from eda import split

USERS_LIMIT = split.val_users()   # nhóm validation (D17); KHÔNG đặt None ở notebook tuning: sẽ đọc cả nhóm test (D17)

pd.set_option("display.max_columns", 30)

R = run(topics=["outliers"], users=USERS_LIMIT)
users, exc = R.users, R.exc
print(f"{len(users)} user, {len(exc):,} chuỗi tìm được")


[20/115] đã đọc tới user 033 (3 giây)
[40/115] đã đọc tới user 063 (5 giây)
[60/115] đã đọc tới user 102 (6 giây)
[80/115] đã đọc tới user 132 (9 giây)
[100/115] đã đọc tới user 163 (12 giây)
[115/115] đã đọc tới user 180 (13 giây)
115 user, 572 chuỗi tìm được


## Bước 1: tập chuỗi

Thêm khoảng ghi điển hình của từng user vào mỗi chuỗi, để phân nhóm ghi dày và thưa ở bước 6. Ngưỡng 5 giây được chọn theo mục 1 (notebook 02): khoảng ghi điển hình của đa số user nằm ở 1–5 giây.


In [10]:
exc = R.exc.merge(users[["user", "median_dt_s"]], on="user", how="left")
GHI_THUA_S = 5                                  # khoảng ghi điển hình > 5 giây là ghi thưa
exc["ghi_thưa"] = exc["median_dt_s"] > GHI_THUA_S

COLS = ["user", "file", "start_utc", "lat_a", "lon_a", "k", "dur_s", "ab_m", "max_dist_m", "max_kmh", "airplane"]
print(f"Tập chuỗi: {len(exc):,} chuỗi, {int(exc['k'].sum()):,} điểm X, "
      f"{exc['user'].nunique()} user; trong nhãn máy bay: {int(exc['airplane'].sum())}")


Tập chuỗi: 572 chuỗi, 152,035 điểm X, 53 user; trong nhãn máy bay: 7


## Bước 2: lưới ứng viên và các chỉ số

Với mỗi ứng viên, bảng tính:
- **số điểm bị xoá**: tổng `k` của các chuỗi bị xoá. Đây là chỉ số chính.
- **điểm tăng thêm**: số điểm bị xoá thêm so với bậc trước. Dùng để tìm điểm gập ở bước 3.
- **số chuỗi, số user, k lớn nhất, thời lượng lớn nhất**: để xem chuỗi bị xoá có quá dài hay không.
- **chuỗi nhãn máy bay**: số chuỗi nằm trong đoạn có nhãn máy bay. Chỉ có dữ liệu cho các user có file nhãn, nên 0 không chứng minh được là không có chuyến bay bị xoá nhầm.

Lưới T gồm 10 giây đến 2 giờ. Lưới N gồm 1, 2, 3, 5, 10, 20, 50.


In [11]:
T_GRID = [10, 30, 60, 120, 300, 600, 1800, 3600, 7200]
N_GRID = [1, 2, 3, 5, 10, 20, 50]

def thong_ke(mask):
    s = exc[mask]
    return {"số_chuỗi": len(s),
            "số_điểm_bị_xoá": int(s["k"].sum()),
            "số_user": s["user"].nunique(),
            "thời_lượng_lớn_nhất": fmt_s(s["dur_s"].max()) if len(s) else "-",
            "k_lớn_nhất": int(s["k"].max()) if len(s) else 0,
            "chuỗi_nhãn_máy_bay": int(s["airplane"].sum())}

bang_T = pd.DataFrame([{"T": fmt_s(T), **thong_ke(exc["dur_s"] <= T)} for T in T_GRID]).set_index("T")
bang_T["điểm_tăng_thêm"] = bang_T["số_điểm_bị_xoá"].diff()

bang_N = pd.DataFrame([{"N": n, **thong_ke(exc["k"] <= n)} for n in N_GRID]).set_index("N")
bang_N["điểm_tăng_thêm"] = bang_N["số_điểm_bị_xoá"].diff()

print("Lọc theo T: xoá chuỗi có thời gian A→B ≤ T")
display(bang_T)
print("Lọc theo N: xoá chuỗi có số điểm k ≤ N")
display(bang_N)


Lọc theo T: xoá chuỗi có thời gian A→B ≤ T


,số_chuỗi,số_điểm_bị_xoá,số_user,thời_lượng_lớn_nhất,k_lớn_nhất,chuỗi_nhãn_máy_bay,điểm_tăng_thêm
T,,,,,,,
10 giây,321,338,4,10 giây,5,0,NaN
30 giây,325,353,5,30 giây,8,0,15.0
1 phút,335,474,10,1 phút,24,1,121.0
2 phút,353,789,15,1.88333 phút,66,2,315.0
5 phút,382,1721,21,4.8 phút,127,4,932.0
10 phút,412,4393,33,10 phút,298,5,2672.0
30 phút,466,12775,45,27.3833 phút,1149,6,8382.0
1 giờ,516,44174,49,59.8333 phút,3099,7,31399.0
2 giờ,548,77544,51,1.94583 giờ,4056,7,33370.0


Lọc theo N: xoá chuỗi có số điểm k ≤ N


,số_chuỗi,số_điểm_bị_xoá,số_user,thời_lượng_lớn_nhất,k_lớn_nhất,chuỗi_nhãn_máy_bay,điểm_tăng_thêm
N,,,,,,,
1,315,315,3,52 giây,1,0,NaN
2,324,333,7,19.7833 phút,2,0,18.0
3,331,354,9,21.1 phút,3,1,21.0
5,342,404,13,21.1 phút,5,2,50.0
10,354,496,16,22.3667 phút,10,2,92.0
20,373,776,22,24.75 phút,20,3,280.0
50,405,1808,28,30.2333 phút,50,5,1032.0


## Bước 3: tìm điểm gập

**Điểm gập** là bậc mà số điểm bị xoá tăng gấp đôi trở lên so với bậc trước. Khi đó, nới ngưỡng thêm một bậc sẽ xoá nhiều điểm hơn hẳn, và rất có thể bắt đầu xoá các đoạn đi thật.

**Ứng viên** là bậc ngay trước điểm gập. Ứng viên của T và N được dùng làm `T_CHON` và `N_CHON` ở bước 5, không cần chọn tay. Hệ số `KNEE_RATIO = 2` là mốc mình chọn để mô tả; bạn có thể đổi và xem ứng viên thay đổi thế nào.


In [12]:
KNEE_RATIO = 2

def diem_gap(bang, grid):
    """Trả về bậc ngay trước điểm gập (giá trị số), hoặc None nếu không có điểm gập."""
    d = bang["điểm_tăng_thêm"]
    ratio = d / d.shift(1)
    labels = list(bang.index)
    hits = [i for i, lab in enumerate(labels) if d[lab] > 0 and ratio[lab] >= KNEE_RATIO]
    if not hits or hits[0] == 0:
        return None
    return grid[hits[0] - 1]

ung_vien_T = diem_gap(bang_T, T_GRID)
ung_vien_N = diem_gap(bang_N, N_GRID)
print(f"Ứng viên T: {fmt_s(ung_vien_T) if ung_vien_T else 'không có điểm gập'}")
print(f"Ứng viên N: {ung_vien_N if ung_vien_N else 'không có điểm gập'}")

# Hai ngưỡng này được dùng cho bước 5; không chọn tay.
T_CHON = ung_vien_T
N_CHON = ung_vien_N

Ứng viên T: 30 giây
Ứng viên N: 3


## Bước 4: so hai cách ở cùng số điểm bị xoá

Bước 3 cho mỗi cách một ứng viên, nhưng hai ứng viên có thể xoá số điểm rất khác nhau. Để so công bằng, bảng này lấy mỗi giá trị N, rồi tìm T xoá số điểm gần nhất với N đó.

Cột **chênh lệch** cho biết hai cách lệch nhau bao nhiêu điểm. Nếu chênh lệch nhỏ, hai cách tương đương về số lượng, và việc chọn phụ thuộc vào các chỉ số khác (thời lượng lớn nhất, nhãn máy bay).


In [13]:
s_sorted = exc.sort_values("dur_s")
cum_k = s_sorted["k"].cumsum().to_numpy()
dur_sorted = s_sorted["dur_s"].to_numpy()

def T_theo_so_diem(b):
    """T nhỏ nhất để số điểm bị xoá (lọc theo T) đạt ít nhất b."""
    i = min(np.searchsorted(cum_k, b), len(cum_k) - 1)
    return float(dur_sorted[i])

rows = []
for n in N_GRID:
    diem_N = thong_ke(exc["k"] <= n)["số_điểm_bị_xoá"]
    T_eq = T_theo_so_diem(diem_N)
    diem_T = thong_ke(exc["dur_s"] <= T_eq)["số_điểm_bị_xoá"]
    rows.append({"N": n, "điểm bị xoá (N)": diem_N, "T tương đương": fmt_s(T_eq),
                 "điểm bị xoá (T)": diem_T, "chênh lệch": diem_T - diem_N})
display(pd.DataFrame(rows).set_index("N"))


,điểm bị xoá (N),T tương đương,điểm bị xoá (T),chênh lệch
N,,,,
1,315,6 giây,330,15
2,333,10 giây,338,5
3,354,48 giây,355,1
5,404,55 giây,422,18
10,496,1.06667 phút,512,16
20,776,1.78333 phút,778,2
50,1808,5.6 phút,1883,75


## Bước 5: bảng 2×2 cho cặp ngưỡng đã chọn

`N_CHON` và `T_CHON` được lấy tự động từ ứng viên ở bước 3, nên không cần đặt tay. Bảng cho biết mỗi chuỗi nằm ở đâu:
- **cả hai bắt:** cả N và T đều xoá chuỗi này.
- **chỉ N bắt:** chuỗi có `k ≤ N` nhưng dài hơn T. Đây là các chuỗi lâu, cần xem thêm.
- **chỉ T bắt:** chuỗi ngắn theo thời gian nhưng có nhiều hơn N điểm.
- **cả hai không bắt:** giữ lại.

Hai bảng dưới liệt kê các chuỗi thuộc hai nhóm "chỉ một cách bắt". Đây là nơi hai cách khác nhau rõ nhất; so các cột `max_kmh` và `max_dist_m` của hai nhóm để thấy khác biệt.


In [14]:
if N_CHON is None or T_CHON is None:
    print("Chưa có ứng viên cho cả hai cách, bỏ qua bước 5.")
    bat_N = pd.Series(False, index=exc.index)
    bat_T = pd.Series(False, index=exc.index)
else:
    bat_N = exc["k"] <= N_CHON
    bat_T = exc["dur_s"] <= T_CHON

nhan_T = fmt_s(T_CHON) if T_CHON else "-"
nhom = {
    f"cả hai bắt (k ≤ {N_CHON} và A→B ≤ {nhan_T})": bat_N & bat_T,
    f"chỉ N bắt (k ≤ {N_CHON}, A→B > {nhan_T})": bat_N & ~bat_T,
    f"chỉ T bắt (A→B ≤ {nhan_T}, k > {N_CHON})": ~bat_N & bat_T,
    "cả hai không bắt": ~bat_N & ~bat_T,
}
bang_22 = pd.DataFrame({
    "số_chuỗi": [int(m.sum()) for m in nhom.values()],
    "số_điểm": [int(exc.loc[m, "k"].sum()) for m in nhom.values()],
}, index=list(nhom.keys()))
display(bang_22)

print("10 chuỗi chỉ N bắt (dài nhất):")
display(exc[bat_N & ~bat_T].nlargest(10, "dur_s")[COLS].round(1))
print("10 chuỗi chỉ T bắt (nhiều điểm nhất):")
display(exc[bat_T & ~bat_N].nlargest(10, "k")[COLS].round(1))

,số_chuỗi,số_điểm
cả hai bắt (k ≤ 3 và A→B ≤ 30 giây),321,331
"chỉ N bắt (k ≤ 3, A→B > 30 giây)",10,23
"chỉ T bắt (A→B ≤ 30 giây, k > 3)",4,22
cả hai không bắt,237,151659


10 chuỗi chỉ N bắt (dài nhất):


,user,file,start_utc,lat_a,lon_a,k,dur_s,ab_m,max_dist_m,max_kmh,airplane
458,083,20090108214517.plt,2009-01-09 10:06:36+00:00,41.7,113.2,3,1266.0,34304.4,31068.3,323.0,False
94,042,20090317033057.plt,2009-03-17 06:01:12+00:00,39.7,116.3,2,1187.0,177.1,2711.5,1839.6,False
497,128,20100124123200.plt,2010-01-24 12:48:00+00:00,40.0,116.3,3,722.0,11282.8,12962.7,3522.9,False
428,062,20081003000004.plt,2008-10-03 07:30:26+00:00,32.2,105.3,2,250.0,4205.3,621288.5,444347.4,False
552,163,20080508002259.plt,2008-05-08 00:48:31+00:00,40.0,116.3,2,138.0,589.6,1610.8,366.4,False
468,115,20080912134336.plt,2008-09-12 17:59:45+00:00,35.5,114.1,3,103.0,4385.6,7999.0,1014.7,False
486,128,20081120003216.plt,2008-11-20 00:51:42+00:00,40.1,116.6,3,80.0,6553.7,4749.8,317.9,True
85,039,20090501030935.plt,2009-05-01 04:26:30+00:00,40.1,116.4,2,75.0,252.0,2571.4,551.7,False
425,062,20081003000004.plt,2008-10-03 06:34:24+00:00,32.0,105.1,1,52.0,901.6,587579.7,423057.4,False
498,128,20100124123200.plt,2010-01-24 13:02:16+00:00,39.8,116.3,2,48.0,260.7,849.0,1528.2,False


10 chuỗi chỉ T bắt (nhiều điểm nhất):


,user,file,start_utc,lat_a,lon_a,k,dur_s,ab_m,max_dist_m,max_kmh,airplane
448,082,20080801051036.plt,2008-08-01 08:44:06+00:00,40.2,116.6,8,22.0,1781.7,1597.4,351.0,False
529,153,20091129234104.plt,2009-11-30 02:30:39+00:00,39.9,116.4,5,6.0,82.7,131.5,473.5,False
532,153,20091129234104.plt,2009-11-30 02:30:56+00:00,39.9,116.4,5,6.0,10.5,122.0,439.2,False
534,153,20091129234104.plt,2009-11-30 02:31:06+00:00,39.9,116.4,4,5.0,263.0,198.3,407.7,False


## Bước 6: kiểm tra theo nhóm ghi dày và thưa

Mỗi bảng đếm số chuỗi bị xoá cho từng nhóm user. Nếu một cách cho kết quả ổn định ở cả hai nhóm, cách đó đáng tin hơn. Nếu nhóm thưa gần như không bị ảnh hưởng ở mọi ngưỡng, ghi đây là điểm chưa giải quyết, vì khoảng ghi thưa làm chuỗi khó xuất hiện đúng dạng quy tắc đang kiểm tra.


In [15]:
def dem_theo_nhom(cot, gia_tri, nhan):
    ket_qua = {}
    for thua, d in exc.groupby("ghi_thưa"):
        ket_qua["ghi thưa (> 5 s)" if thua else "ghi dày (≤ 5 s)"] = [int((d[cot] <= g).sum()) for g in gia_tri]
    return pd.DataFrame(ket_qua, index=nhan).T

print("Số chuỗi bị xoá, lọc theo T:")
display(dem_theo_nhom("dur_s", T_GRID, [fmt_s(T) for T in T_GRID]))
print("Số chuỗi bị xoá, lọc theo N:")
display(dem_theo_nhom("k", N_GRID, N_GRID))


Số chuỗi bị xoá, lọc theo T:


,10 giây,30 giây,1 phút,2 phút,5 phút,10 phút,30 phút,1 giờ,2 giờ
ghi dày (≤ 5 s),321,325,335,352,381,410,461,511,542
ghi thưa (> 5 s),0,0,0,1,1,2,5,5,6


Số chuỗi bị xoá, lọc theo N:


,1,2,3,5,10,20,50
ghi dày (≤ 5 s),315,324,331,340,352,370,401
ghi thưa (> 5 s),0,0,0,2,2,3,4


## Bước 7: so sánh N = 3 và N = 5

N = 5 xoá thêm các chuỗi có k = 4 hoặc 5 so với N = 3. Bước này xem riêng các chuỗi đó có giống lỗi (nhảy xa, tốc độ vô lý, A và B gần nhau) hay giống di chuyển thật, và so với nhóm k ≤ 3.

In [16]:
A, B = 3, 5
nhom_A = exc[exc["k"] <= A]
them = exc[(exc["k"] > A) & (exc["k"] <= B)]
print(f"N = {A}: {len(nhom_A)} chuỗi, {int(nhom_A['k'].sum())} điểm, {nhom_A['user'].nunique()} user, nhãn máy bay {int(nhom_A['airplane'].sum())}")
print(f"N = {B}: thêm {len(them)} chuỗi, {int(them['k'].sum())} điểm, {them['user'].nunique()} user, nhãn máy bay {int(them['airplane'].sum())}")

def tom_tat(d):
    return pd.Series({"số_chuỗi": len(d), "k trung vị": d["k"].median(), "max_kmh trung vị": d["max_kmh"].median(),
                      "max_dist_m trung vị": d["max_dist_m"].median(), "ab_m trung vị": d["ab_m"].median(),
                      "dur_s trung vị": d["dur_s"].median(), "% ghi thưa": 100 * d["ghi_thưa"].mean(), "số user": d["user"].nunique()})
display(pd.DataFrame({f"k ≤ {A}": tom_tat(nhom_A), f"{A} < k ≤ {B} (chỉ N = {B} xoá)": tom_tat(them)}).round(1))
print(f"Các chuỗi chỉ N = {B} xoá:")
display(them.sort_values(["k", "max_kmh"], ascending=[False, False])[COLS].round(1))

N = 3: 331 chuỗi, 354 điểm, 9 user, nhãn máy bay 1
N = 5: thêm 11 chuỗi, 50 điểm, 7 user, nhãn máy bay 1


,k ≤ 3,3 < k ≤ 5 (chỉ N = 5 xoá)
số_chuỗi,331.0,11.0
k trung vị,1.0,5.0
max_kmh trung vị,1546312.6,473.5
max_dist_m trung vị,857116.1,2477.9
ab_m trung vị,132.3,906.3
dur_s trung vị,5.0,107.0
% ghi thưa,0.0,18.2
số user,9.0,7.0


Các chuỗi chỉ N = 5 xoá:


,user,file,start_utc,lat_a,lon_a,k,dur_s,ab_m,max_dist_m,max_kmh,airplane
567,176,20071207031942.plt,2007-12-07 04:23:41+00:00,40.0,116.6,5,327.0,8244.3,49936.5,40503.9,False
53,017,20081122101352.plt,2008-11-22 21:44:38+00:00,40.0,116.4,5,122.0,1869.1,1936.0,1150.3,False
496,128,20090921001453.plt,2009-09-21 00:17:17+00:00,40.1,116.3,5,107.0,906.3,2477.9,557.5,False
529,153,20091129234104.plt,2009-11-30 02:30:39+00:00,39.9,116.4,5,6.0,82.7,131.5,473.5,False
532,153,20091129234104.plt,2009-11-30 02:30:56+00:00,39.9,116.4,5,6.0,10.5,122.0,439.2,False
488,128,20081125080000.plt,2008-11-25 09:13:25+00:00,23.4,113.3,5,60.0,4990.1,4116.9,323.5,True
464,101,20080202025001.plt,2008-02-02 06:48:53+00:00,32.5,120.1,4,150.0,284.6,2675.6,663.7,False
465,101,20080202105001.plt,2008-02-02 14:48:53+00:00,32.5,120.1,4,150.0,284.6,2675.6,663.7,False
112,062,20080823095428.plt,2008-08-23 10:24:52+00:00,40.0,116.3,4,195.0,3592.5,11261.3,414.1,False
534,153,20091129234104.plt,2009-11-30 02:31:06+00:00,39.9,116.4,4,5.0,263.0,198.3,407.7,False


## Tiêu chí chốt ngưỡng

Ghi trước khi xem kết quả cuối cùng. Các tiêu chí dưới đây là đề xuất, bạn có thể sửa.

1. **Cách và giá trị:** chọn cách (N hoặc T) có điểm gập rõ, rồi lấy ứng viên ngay trước điểm gập (bước 3).
2. **Cùng số điểm:** nếu hai cách cho số điểm bị xoá gần nhau (bước 4), chọn cách có thời lượng lớn nhất của chuỗi bị xoá thấp hơn.
3. **Máy bay:** không có chuỗi nhãn máy bay bị xoá trong các user có nhãn (bước 2 và bước 5).
4. **Nhóm ghi:** cả nhóm dày và thưa có phản ứng tương tự (bước 6). Nếu không, ghi rõ là chưa giải quyết.

**Quyết định:** xoá chuỗi có `k ≤ 5` điểm (cách N, N = 5), không thêm điều kiện thời gian (A6, Duy chốt 2026-11-07).

Căn cứ trên validation (115 user, 572 chuỗi, 152.035 điểm, 53 user):
- **Cách và điểm gập (tiêu chí 1):** quy tắc điểm gập cho N = 3 và T = 30 giây (với N, bậc N = 5 tăng thêm 50 điểm so với 21 điểm ở bậc N = 3, gấp 2,4 lần). Ở cùng số điểm bị xoá (bước 4) hai cách chênh nhau 1–18 điểm khi N ≤ 5 nên tương đương số lượng; cách N được chọn vì đếm điểm không phụ thuộc tần suất ghi (T gần như không bắt chuỗi nào ở nhóm ghi thưa, bước 6: 0 chuỗi đến T = 1 phút).
- **N = 5 thay vì 3:** vùng N = 3–10 khá phẳng. N = 5 xoá thêm 11 chuỗi và 50 điểm so với N = 3; trong 11 chuỗi thêm đó 10 là lỗi (tốc độ 323 đến 40.504 km/h, không phải máy bay, tàu tối đa 237 km/h) và 1 là máy bay thật (user 128).
- **Máy bay (tiêu chí 3):** không đạt hoàn toàn. N = 5 xoá 2 chuỗi trong đoạn nhãn máy bay (N = 3 xoá 1), cả hai ở user 128. Chuỗi dài hơn 5 điểm được giữ.
- **Nhóm ghi (tiêu chí 4):** chưa giải quyết đủ: nhóm ghi thưa gần như không bị ảnh hưởng ở N ≤ 3 (0 chuỗi) và chỉ 2 chuỗi ở N = 5.

Đây là quy ước trong vùng phẳng, không phải điểm tối ưu.